In [14]:
import tensorflow as tf
import pandas as pd
import glob
import os

INPUT_TFRECORD_DIR = "/pdo/astronet-data/data/tfrecords/dec2025_cad_scat_v5_aug/10x_0p1/tfrecords-vetting-v01-tois-triageJs-nocentroid-dec2025-val/"
INPUT_FILES = sorted(
    f for f in glob.glob(os.path.join(INPUT_TFRECORD_DIR, "*"))
    if os.path.isfile(f)
)

OUTPUT_TFRECORD = "/pdo/astronet-data/data/tfrecords/dec2025_cad_scat_v5_aug/10x_0p1_with_new_features/val/01"
FEATURE_CSV = "/pdo/astronet-data/data/labels/tces-vetting-v01-tois-triageJs-nocentroid-april2025-all-qlp-mast-data.csv"

# Load and prepare lookup table once
feature_df = pd.read_csv(FEATURE_CSV, index_col=None)
feature_df = feature_df.reset_index()

# Normalize astro_id type
feature_df["astro_id"] = pd.to_numeric(feature_df["astro_id"], errors="coerce")
feature_df = feature_df.dropna(subset=["astro_id"]).copy()
feature_df["astro_id"] = feature_df["astro_id"].astype(int)

# Keep one row per astro_id if duplicates exist
feature_df = feature_df.drop_duplicates(subset=["astro_id"], keep="first")

# Fast lookup by astro_id
feature_df = feature_df.set_index("astro_id")

FEATURE_SPECS = {
    "star_t_eff": {
        "mean": 6265.31676,
    },
    "snr": {
        "mean": 66.404972,
    },
    "t12_t14": {
        "mean": 0.286041,
    },
    "numcont": {
        "mean": 583.509533,
    },
}


def set_float_feature(example: tf.train.Example, name: str, value: float) -> None:
    example.features.feature[name].float_list.value[:] = [float(value)]


def add_new_field(serialized_example: tf.Tensor) -> bytes:
    example = tf.train.Example()
    example.ParseFromString(serialized_example.numpy())

    astro_id = int(example.features.feature["astro_id"].int64_list.value[0])

    if astro_id in feature_df.index:
        row = feature_df.loc[astro_id]
    else:
        row = None

    for feature_name, spec in FEATURE_SPECS.items():
        present_name = f"{feature_name}_present"
        default_value = spec["mean"]

        if row is None or pd.isna(row[feature_name]):
            value = default_value
            present = 0.0
        else:
            value = float(row[feature_name])
            present = 1.0

        set_float_feature(example, feature_name, value)
        set_float_feature(example, present_name, present)

    return example.SerializeToString()


def rewrite_tfrecord(input_files, output_path: str) -> None:
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    dataset = tf.data.TFRecordDataset(input_files)

    count = 0
    with tf.io.TFRecordWriter(output_path) as writer:
        for raw_record in dataset:
            writer.write(add_new_field(raw_record))
            count += 1
            if count % 10000 == 0:
                print(f"Processed {count} examples")

    print(f"Wrote {count} examples to: {output_path}")


print('Rewriting...')
rewrite_tfrecord(INPUT_FILES, OUTPUT_TFRECORD)

Rewriting...


2026-03-26 22:49:09.432737: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [50]
	 [[{{node Placeholder/_0}}]]


Wrote 1547 examples to: /pdo/astronet-data/data/tfrecords/dec2025_cad_scat_v5_aug/10x_0p1_with_new_features/val/01


In [2]:
# Compute statistics
feature_spec = {
    "astro_id": tf.io.FixedLenFeature([], tf.int64),
}

dataset = tf.data.TFRecordDataset(INPUT_FILES)
astro_ids = []

for raw_record in dataset:
    parsed = tf.io.parse_single_example(raw_record, feature_spec)
    astro_id = int(parsed["astro_id"].numpy())
    astro_ids.append(astro_id)

print(f"Total IDs: {len(astro_ids)}")
print(f"Unique IDs: {len(set(astro_ids))}")

2026-03-26 22:30:12.753366: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.
2026-03-26 22:30:13.182607: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [550]
	 [[{{node Placeholder/_0}}]]


Total IDs: 136125
Unique IDs: 12375


In [9]:
feature_df = feature_df.reset_index()
astro_ids = set(astro_ids)
df_filtered = feature_df[feature_df["astro_id"].isin(astro_ids)]
feature = "star_t_eff"

valid = df_filtered[feature].dropna()

mean = valid.mean()
std = valid.std()

print(f"Mean ({feature}): {mean}")
print(f"Std ({feature}): {std}")
print(f"Non-null count: {len(valid)}")
print(f"Coverage: {len(valid) / len(df_filtered):.3f}")

Mean (star_t_eff): 6265.316761779661
Std (star_t_eff): 2046.9632529426915
Non-null count: 3540
Coverage: 0.289


In [39]:
import numpy as np
path = "/pdo/astronet-data/models/vetting/experimental/dimond/20260325/dimond/AstroCNNModelVetting_cshallue_20260325_231001/evaluation/train_label.npy"

arr = np.load(path)
print(arr.shape)

(5120, 4)
